# Compute residual norm coefficients

This notebook computes the residual norm coefficients as part of the variable weights.

In [1]:
import os
import yaml
import copy
import numpy as np
import xarray as xr

In [2]:
from scipy.stats import gmean

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

## ERA5

In [4]:
# get variable information from data_preprocessing/config
config_name = os.path.realpath('data_config_ERA5.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [5]:
N_levels = 11

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/all_in_one/'
ds_example = xr.open_zarr(base_dir+'ERA5_GP_1980.zarr')
level = np.array(ds_example['level'])

In [6]:
# get variable names
varnames = list(conf['residual'].keys())
varnames = varnames[:-5] # remove save_loc and others

varname_surf = list(set(varnames) - set(['U', 'V', 'T', 'Q']))
varname_upper = ['U', 'V', 'T', 'Q']

In [7]:
# collect computed mean and variance values
# See "qsub_STEP01_compute_mean_std.ipynb"
MEAN_values = {}
STD_values = {}

for varname in varname_surf:
    save_name = conf['residual']['save_loc'] + '{}_mean_std_{}.npy'.format(
        conf['residual']['prefix'], varname)
    mean_std = np.load(save_name)
    MEAN_values[varname] = mean_std[0]
    STD_values[varname] = mean_std[1]

for varname in varname_upper:

    # -------------------------------------------- #
    # allocate all levels
    mean_std_all_levels = np.empty((2, N_levels))
    mean_std_all_levels[...] = np.nan
    
    for i_level in range(N_levels):
        save_name = conf['residual']['save_loc'] + '{}_level{}_mean_std_{}.npy'.format(
            conf['residual']['prefix'], i_level, varname)
        mean_std = np.load(save_name)
        mean_std_all_levels[:, i_level] = mean_std

    # -------------------------------------------- #
    # save
    MEAN_values[varname] = np.copy(mean_std_all_levels[0, :])
    STD_values[varname] = np.copy(mean_std_all_levels[1, :])

keys_to_drop = ['TCC', 'SKT', 'land_sea_CI_mask'] # <---------------- some variables are not used in the paper
MEAN_values = {k: v for k, v in MEAN_values.items() if k not in keys_to_drop}
STD_values = {k: v for k, v in STD_values.items() if k not in keys_to_drop}

In [8]:
# separate upper air (list) and surf (float) std values
std_val_all = list(STD_values.values())
std_val_surf = np.array(std_val_all[:-4])
std_val_upper = std_val_all[-4:]

# combine
std_concat = np.concatenate([std_val_surf]+ std_val_upper)

# geometrical mean (not used)
std_g = gmean(np.sqrt(std_concat))

### Save residual coef as a file

In [9]:
# ------------------------------------------------------- #
# create xr.DataArray for std
ds_std_6h = xr.Dataset(coords={"level": level})

for varname, data in STD_values.items():
    data = np.sqrt(data) / std_g # <--- var to std and divided by std_g
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["level",],
            coords={"level": level},
            name=varname,
        )
        ds_std_6h[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_std_6h[varname] = data_array

In [10]:
# ds_std_6h.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/ERA5_6h_residual_1980_2019.nc')

## WRF

In [4]:
# get variable information from data_preprocessing/config
config_name = os.path.realpath('data_config_WRF.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [5]:
N_levels = 16

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/all_in_one/'
ds_example = xr.open_zarr(base_dir+'C404_GP_1980.zarr')
level = np.array(ds_example['bottom_top'])

In [6]:
# get variable names
varnames = list(conf['residual'].keys())
varnames = varnames[:-5] # remove save_loc and others

varname_surf = list(set(varnames) - set(['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q', 'WRF_P']))
varname_upper = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q', 'WRF_P']

In [7]:
MEAN_values = {}
STD_values = {}

for varname in varname_surf:
    save_name = conf['residual']['save_loc'] + '{}_mean_std_{}.npy'.format(
        conf['residual']['prefix'], varname)
    mean_std = np.load(save_name)
    MEAN_values[varname] = mean_std[0]
    STD_values[varname] = mean_std[1]

for varname in varname_upper:

    # -------------------------------------------- #
    # allocate all levels
    mean_std_all_levels = np.empty((2, N_levels))
    mean_std_all_levels[...] = np.nan
    
    for i_level in range(N_levels):
        save_name = conf['residual']['save_loc'] + '{}_level{}_mean_std_{}.npy'.format(
            conf['residual']['prefix'], i_level, varname)
        mean_std = np.load(save_name)
        mean_std_all_levels[:, i_level] = mean_std

    # -------------------------------------------- #
    # save
    MEAN_values[varname] = np.copy(mean_std_all_levels[0, :])
    STD_values[varname] = np.copy(mean_std_all_levels[1, :])

In [8]:
# separate upper air (list) and surf (float) std values
std_val_all = list(STD_values.values())
std_val_surf = np.array(std_val_all[:-5])
std_val_upper = std_val_all[-5:]

# combine
std_concat = np.concatenate([std_val_surf]+ std_val_upper)

# geometrical mean (not used)
std_g = gmean(np.sqrt(std_concat))

In [9]:
# ------------------------------------------------------- #
# create xr.DataArray for std

# use the same level coord as mean
ds_std_6h = xr.Dataset(coords={'bottom_top': level})

for varname, data in STD_values.items():
    data = np.sqrt(data) / std_g
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["bottom_top",],
            coords={"bottom_top": level},
            name=varname,
        )
        ds_std_6h[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_std_6h[varname] = data_array

In [12]:
# ds_std_6h.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_6h_residual_1980_2019_16lev.nc')

In [13]:
ds_std_6h

<xarray.Dataset>
Dimensions:              (bottom_top: 16)
Coordinates:
  * bottom_top           (bottom_top) float32 0.0 1.0 2.0 3.0 ... 13.0 14.0 15.0
Data variables: (12/18)
    WRF_SRH03            float64 3.028
    WRF_precip           float64 9.588
    WRF_OLR              float64 4.239
    WRF_TD2              float64 0.7996
    WRF_SP               float64 0.1191
    WRF_evapor           float64 2.811
    ...                   ...
    WRF_MLCAPE           float64 2.321
    WRF_U                (bottom_top) float64 3.284 2.875 2.612 ... 0.877 1.231
    WRF_V                (bottom_top) float64 2.749 2.355 2.167 ... 1.404 3.146
    WRF_T                (bottom_top) float64 0.8337 0.74 0.6957 ... 1.077 1.049
    WRF_Q                (bottom_top) float64 0.8854 0.9998 ... 1.757 2.348
    WRF_P                (bottom_top) float64 0.1189 0.119 ... 0.1325 0.315